In [6]:
# =========================================================================
# CREDIT RISK DEMO
# LOAD PKL + MANUALLY ENTER ONE CUSTOMER + PREDICT CREDIT RISK
# =========================================================================

import os
import joblib
import numpy as np
import pandas as pd


# =========================================================================
# 1. LOAD PKL
# =========================================================================

BASE_DIR = r"D:\Thesis\8-PKL-file"

PKL_PATH = os.path.join(
    BASE_DIR,
    "credit_risk_package.pkl"
)

if not os.path.exists(PKL_PATH):
    raise FileNotFoundError(
        f"PKL file not found:\n{PKL_PATH}"
    )

credit_risk_package = joblib.load(PKL_PATH)


# =========================================================================
# 2. LOAD MODEL INFORMATION
# =========================================================================

final_model = credit_risk_package["model"]

scalers = credit_risk_package["scalers"]

log_features = credit_risk_package["log_features"]

scale_features = credit_risk_package["scale_features"]

binary_features = credit_risk_package["binary_features"]

categorical_features = credit_risk_package["categorical_features"]

final_features = credit_risk_package["final_features"]

training_encoded_columns = (
    credit_risk_package["training_encoded_columns"]
)

best_threshold = credit_risk_package["best_threshold"]

t1 = credit_risk_package["t1"]

t2 = credit_risk_package["t2"]

occupation_values = (
    credit_risk_package["occupation_values"]
)

product_type_values = (
    credit_risk_package["product_type_values"]
)

loan_intent_values = (
    credit_risk_package["loan_intent_values"]
)


# =========================================================================
# 3. INPUT FUNCTIONS
# =========================================================================

def ask_text(prompt):
    while True:
        value = input(prompt).strip()

        if value:
            return value

        print("Please enter a value.")


def ask_int(prompt, minimum=None, maximum=None):
    while True:
        try:
            value = int(input(prompt).strip())

            if minimum is not None and value < minimum:
                print(f"Value must be at least {minimum}.")
                continue

            if maximum is not None and value > maximum:
                print(f"Value must be at most {maximum}.")
                continue

            return value

        except ValueError:
            print("Please enter a valid integer.")


def ask_float(prompt, minimum=None, maximum=None):
    while True:
        try:
            value = float(input(prompt).strip())

            if minimum is not None and value < minimum:
                print(f"Value must be at least {minimum}.")
                continue

            if maximum is not None and value > maximum:
                print(f"Value must be at most {maximum}.")
                continue

            return value

        except ValueError:
            print("Please enter a valid number.")


def ask_category(prompt, allowed_values):
    print(f"\n{prompt}")

    for i, value in enumerate(allowed_values, start=1):
        print(f"  {i}. {value}")

    while True:
        try:
            choice = int(
                input("Choose number: ").strip()
            )

            if 1 <= choice <= len(allowed_values):
                return allowed_values[choice - 1]

            print("Please choose one of the listed numbers.")

        except ValueError:
            print("Please enter the number of your choice.")


# =========================================================================
# 4. START DEMO
# =========================================================================

print("\n" + "=" * 75)
print("             CREDIT RISK DECISION SUPPORT SYSTEM")
print("=" * 75)
print("Please enter the customer information.")
print("=" * 75)


# =========================================================================
# 5. ENTER CUSTOMER INFORMATION
# =========================================================================

customer_id = ask_text(
    "\nCustomer ID: "
)


age = ask_int(
    "Age: ",
    minimum=18,
    maximum=100
)


occupation_status = ask_category(
    "Occupation Status:",
    occupation_values
)


years_employed = ask_float(
    "Years Employed: ",
    minimum=0,
    maximum=60
)


annual_income = ask_float(
    "Annual Income: ",
    minimum=0
)


credit_score = ask_float(
    "Credit Score: ",
    minimum=300,
    maximum=850
)


credit_history_years = ask_float(
    "Credit History Years: ",
    minimum=0,
    maximum=80
)


savings_assets = ask_float(
    "Savings / Assets: ",
    minimum=0
)


current_debt = ask_float(
    "Current Debt: ",
    minimum=0
)


defaults_on_file = ask_int(
    "Defaults on File: ",
    minimum=0
)


delinquencies_last_2yrs = ask_int(
    "Delinquencies in Last 2 Years: ",
    minimum=0
)


derogatory_marks = ask_int(
    "Derogatory Marks: ",
    minimum=0
)


product_type = ask_category(
    "Product Type:",
    product_type_values
)


loan_intent = ask_category(
    "Loan Intent:",
    loan_intent_values
)


loan_amount = ask_float(
    "Loan Amount: ",
    minimum=0
)


interest_rate = ask_float(
    "Interest Rate (%): ",
    minimum=0,
    maximum=100
)


payment_to_income_ratio = ask_float(
    "Payment-to-Income Ratio: ",
    minimum=0
)


# =========================================================================
# 6. CALCULATE RATIOS AUTOMATICALLY
# =========================================================================

if annual_income <= 0:
    raise ValueError(
        "Annual income must be greater than zero."
    )


debt_to_income_ratio = (
    current_debt / annual_income
)


loan_to_income_ratio = (
    loan_amount / annual_income
)


# =========================================================================
# 7. CREATE CUSTOMER DICTIONARY
# =========================================================================

customer = {

    "customer_id": customer_id,

    "age": age,

    "occupation_status": occupation_status,

    "years_employed": years_employed,

    "annual_income": annual_income,

    "credit_score": credit_score,

    "credit_history_years": credit_history_years,

    "savings_assets": savings_assets,

    "current_debt": current_debt,

    "defaults_on_file": defaults_on_file,

    "delinquencies_last_2yrs":
        delinquencies_last_2yrs,

    "derogatory_marks":
        derogatory_marks,

    "product_type": product_type,

    "loan_intent": loan_intent,

    "loan_amount": loan_amount,

    "interest_rate": interest_rate,

    "debt_to_income_ratio":
        debt_to_income_ratio,

    "loan_to_income_ratio":
        loan_to_income_ratio,

    "payment_to_income_ratio":
        payment_to_income_ratio
}


# =========================================================================
# 8. DATA VALIDATION
# =========================================================================

validation_errors = []


# Age
if age < 18 or age > 100:
    validation_errors.append(
        "Age must be between 18 and 100."
    )


# Employment
if years_employed > age - 18:
    validation_errors.append(
        "Years employed exceeds the possible employment period."
    )


# Credit history
if credit_history_years > age - 18:
    validation_errors.append(
        "Credit history exceeds the possible period."
    )


# Income
if annual_income <= 0:
    validation_errors.append(
        "Annual income must be greater than zero."
    )


# Financial variables
if savings_assets < 0:
    validation_errors.append(
        "Savings/assets cannot be negative."
    )


if current_debt < 0:
    validation_errors.append(
        "Current debt cannot be negative."
    )


if loan_amount < 0:
    validation_errors.append(
        "Loan amount cannot be negative."
    )


# Ratios
if debt_to_income_ratio < 0:
    validation_errors.append(
        "Debt-to-income ratio cannot be negative."
    )


if loan_to_income_ratio < 0:
    validation_errors.append(
        "Loan-to-income ratio cannot be negative."
    )


if payment_to_income_ratio < 0:
    validation_errors.append(
        "Payment-to-income ratio cannot be negative."
    )


# =========================================================================
# 9. STOP IF VALIDATION FAILS
# =========================================================================

if validation_errors:

    print("\n" + "=" * 75)
    print("DATA VALIDATION FAILED")
    print("=" * 75)

    for error in validation_errors:
        print(f"❌ {error}")

    print("=" * 75)

    raise SystemExit


# =========================================================================
# 10. CREATE DATAFRAME
# =========================================================================

raw_customer = pd.DataFrame(
    [customer]
)


# =========================================================================
# 11. REMOVE ID FROM MODEL INPUT
# =========================================================================

X_new = raw_customer.drop(
    columns=["customer_id"]
).copy()


# =========================================================================
# 12. LOG TRANSFORMATION
# =========================================================================

for col in log_features:

    X_new[
        f"{col}_log"
    ] = np.log1p(
        X_new[col]
    )


# =========================================================================
# 13. SCALE ORIGINAL FEATURES
# =========================================================================

for col in scale_features:

    scaler = scalers[col]

    X_new[
        f"{col}_scaled"
    ] = scaler.transform(
        X_new[[col]]
    )


# =========================================================================
# 14. SCALE LOG FEATURES
# =========================================================================

log_scaled_features = [
    f"{col}_log"
    for col in log_features
]


for col in log_scaled_features:

    scaler = scalers[col]

    X_new[
        f"{col}_scaled"
    ] = scaler.transform(
        X_new[[col]]
    )


# =========================================================================
# 15. BINARY FEATURES
# =========================================================================

for col in binary_features:

    X_new[
        f"{col}_binary"
    ] = (
        X_new[col] > 0
    ).astype(int)


# =========================================================================
# 16. ONE-HOT ENCODING
# =========================================================================

X_new_encoded = pd.get_dummies(
    X_new,
    columns=categorical_features,
    drop_first=True
)


# =========================================================================
# 17. MATCH TRAINING COLUMNS EXACTLY
# =========================================================================

X_new_encoded = (
    X_new_encoded
    .reindex(
        columns=training_encoded_columns,
        fill_value=0
    )
)


# =========================================================================
# 18. DROP ORIGINAL FEATURES
# =========================================================================

features_to_drop = (
    log_features
    +
    scale_features
    +
    binary_features
)


X_new_encoded = (
    X_new_encoded
    .drop(
        columns=features_to_drop,
        errors="ignore"
    )
)


# =========================================================================
# 19. SELECT FINAL MODEL FEATURES
# =========================================================================

X_new_final = (
    X_new_encoded[
        final_features
    ]
)


# =========================================================================
# 20. MODEL PREDICTION
# =========================================================================

risk_probability = float(

    final_model
    .predict_proba(
        X_new_final
    )[0, 1]

)


# =========================================================================
# 21. THREE-ZONE DECISION
# =========================================================================

if risk_probability < t1:

    risk_level = "LOW RISK"

    decision = "SAFE - AUTO ACCEPT"

    explanation = (
        "The predicted risk probability "
        "is below the lower threshold."
    )


elif risk_probability < t2:

    risk_level = "MEDIUM / UNCERTAIN RISK"

    decision = "HUMAN REVIEW"

    explanation = (
        "The predicted risk probability "
        "falls inside the uncertainty zone."
    )


else:

    risk_level = "HIGH RISK"

    decision = "RISKY - AUTO REJECT"

    explanation = (
        "The predicted risk probability "
        "is above the upper threshold."
    )


# =========================================================================
# 22. DISPLAY CUSTOMER
# =========================================================================

print("\n" + "=" * 75)
print("CUSTOMER INFORMATION")
print("=" * 75)

print(
    f"Customer ID:              {customer_id}"
)

print(
    f"Age:                      {age}"
)

print(
    f"Occupation:               {occupation_status}"
)

print(
    f"Years Employed:           {years_employed}"
)

print(
    f"Annual Income:            {annual_income:,.2f}"
)

print(
    f"Credit Score:             {credit_score:.0f}"
)

print(
    f"Credit History:           {credit_history_years:.1f} years"
)

print(
    f"Savings / Assets:         {savings_assets:,.2f}"
)

print(
    f"Current Debt:             {current_debt:,.2f}"
)

print(
    f"Defaults on File:         {defaults_on_file}"
)

print(
    f"Delinquencies:            {delinquencies_last_2yrs}"
)

print(
    f"Derogatory Marks:         {derogatory_marks}"
)

print(
    f"Product Type:             {product_type}"
)

print(
    f"Loan Intent:              {loan_intent}"
)

print(
    f"Loan Amount:              {loan_amount:,.2f}"
)

print(
    f"Interest Rate:            {interest_rate:.2f}%"
)

print(
    f"Debt-to-Income Ratio:     {debt_to_income_ratio:.4f}"
)

print(
    f"Loan-to-Income Ratio:     {loan_to_income_ratio:.4f}"
)

print(
    f"Payment-to-Income Ratio:  {payment_to_income_ratio:.4f}"
)


# =========================================================================
# 23. DISPLAY RESULT
# =========================================================================

print("\n" + "=" * 75)
print("CREDIT RISK ASSESSMENT")
print("=" * 75)

print(
    f"Risk Probability:         "
    f"{risk_probability:.2%}"
)

print(
    f"Lower Threshold (t1):     "
    f"{t1:.2%}"
)

print(
    f"Upper Threshold (t2):     "
    f"{t2:.2%}"
)

print(
    f"Risk Level:               "
    f"{risk_level}"
)

print(
    f"Final Decision:           "
    f"{decision}"
)

print()

print(
    explanation
)

print("=" * 75)


# =========================================================================
# 24. CLEAR RESULT MESSAGE
# =========================================================================

if risk_probability < t1:

    print()
    print("🟢 SAFE CUSTOMER")
    print("Recommendation: AUTO-ACCEPT")


elif risk_probability < t2:

    print()
    print("🟡 HUMAN REVIEW REQUIRED")
    print("Recommendation: SEND TO HUMAN CREDIT ANALYST")


else:

    print()
    print("🔴 RISKY CUSTOMER")
    print("Recommendation: AUTO-REJECT")


print("\n" + "=" * 75)
print("Prediction completed.")
print("=" * 75)


             CREDIT RISK DECISION SUPPORT SYSTEM
Please enter the customer information.



Customer ID:  demo003
Age:  45



Occupation Status:
  1. Employed
  2. Self-Employed
  3. Student


Choose number:  3
Years Employed:  5
Annual Income:  20000
Credit Score:  308
Credit History Years:  2
Savings / Assets:  0
Current Debt:  10000
Defaults on File:  0
Delinquencies in Last 2 Years:  1
Derogatory Marks:  0



Product Type:
  1. Credit Card
  2. Line of Credit
  3. Personal Loan


Choose number:  3



Loan Intent:
  1. Business
  2. Debt Consolidation
  3. Education
  4. Home Improvement
  5. Medical
  6. Personal


Choose number:  6
Loan Amount:  10000
Interest Rate (%):  10
Payment-to-Income Ratio:  8



CUSTOMER INFORMATION
Customer ID:              demo003
Age:                      45
Occupation:               Student
Years Employed:           5.0
Annual Income:            20,000.00
Credit Score:             308
Credit History:           2.0 years
Savings / Assets:         0.00
Current Debt:             10,000.00
Defaults on File:         0
Delinquencies:            1
Derogatory Marks:         0
Product Type:             Personal Loan
Loan Intent:              Personal
Loan Amount:              10,000.00
Interest Rate:            10.00%
Debt-to-Income Ratio:     0.5000
Loan-to-Income Ratio:     0.5000
Payment-to-Income Ratio:  8.0000

CREDIT RISK ASSESSMENT
Risk Probability:         0.00%
Lower Threshold (t1):     20.00%
Upper Threshold (t2):     60.00%
Risk Level:               LOW RISK
Final Decision:           SAFE - AUTO ACCEPT

The predicted risk probability is below the lower threshold.

🟢 SAFE CUSTOMER
Recommendation: AUTO-ACCEPT

Prediction completed.
